# JobKB — inspection & QA
Quick spot-checks over the `canonical/` outputs of `run_pipeline.py`.

In [1]:
import os, pandas as pd
pd.set_option('display.max_colwidth', 80)
CAN = os.path.join('..', 'canonical')
def load(name): return pd.read_csv(os.path.join(CAN, name), dtype=str, keep_default_na=False)
occ   = load('occupations.csv')
skl   = load('skills.csv')
align = load('concept_alignments.csv')
cocc  = load('canonical_occupations.csv')
cskl  = load('canonical_skills.csv')
prov  = load('provenance.csv')
prov

,entity_id,source,source_version,retrieved_at,retrieval_method,notes
0,ISCO,ISCO,ISCO-08,2026-07-19T09:13:22+00:00,official_en_csv,"21 groups, 19 edges (IT branches 25/35)"
1,ESCO,ESCO,ESCO v1.2 (en),2026-07-19T09:13:25+00:00,official_en_csv,"97 occ, 1158 skills, 5959 relations"
2,ONET,ONET,O*NET 28.x,2026-07-19T09:13:41+00:00,official_en_csv,"31 occ, 1469 skills, 6619 relations"
3,NOC,NOC,NOC 2021 v1.0,2026-07-19T09:13:48+00:00,official_en_fr_csv,15 IT unit-group occupations (bilingual)
4,ROME,ROME,ROME v461,2026-07-19T09:13:53+00:00,official_fr_csv,"96 M18 metiers, 2162 skills, 5296 relations"
5,ESCO_SKILLS,ESCO_SKILLS,ESCO v1.2 (en),2026-07-19T09:13:55+00:00,derived,"95 soft, 209 groups, 1767 skill edges"
6,ALIGNMENT,ALIGNMENT,-,2026-07-19T10:33:53+00:00,embed:st+nli:on,"26643 alignments, 58 grafts"
7,MERGE,MERGE,-,2026-07-19T10:33:53+00:00,connected_components(exactMatch),"232 canonical occ (6 merged), 4818 canonical skills (63 merged)"


In [2]:
# Counts per source
print('Occupations by source:'); print(occ['source'].value_counts())
print('\nSkills by source:');     print(skl['source'].value_counts())
print('\nEN label coverage (real occ):',
      (occ[occ.occupation_type!='isco_group'].pref_label_en!='').sum(),
      '/', (occ.occupation_type!='isco_group').sum())

Occupations by source:
source
ESCO    97
ROME    96
ONET    31
ISCO    21
NOC     15
Name: count, dtype: int64

Skills by source:
source
ROME    2162
ONET    1469
ESCO    1461
Name: count, dtype: int64

EN label coverage (real occ): 143 / 239


In [3]:
# De-duplication: canonical occupations that merged >1 source entity
cocc['n_members'] = cocc.member_entity_ids.str.split(' | ').map(len)
merged = cocc[cocc.n_members > 1].sort_values('n_members', ascending=False)
print(f'{len(merged)} multi-source canonical occupations of {len(cocc)} total')
merged[['primary_label_en','primary_label_fr','isco_code','sources','n_members']].head(25)

6 multi-source canonical occupations of 232 total


,primary_label_en,primary_label_fr,isco_code,sources,n_members
20,data scientist,scientifique des données,2511,ESCO | NOC | ONET,5
16,data engineer,ingénieur/ingénieure de données,2511,ESCO | ROME,3
24,cloud architect,architecte cloud,2512,ESCO | ROME,3
59,database administrator,administrateur de base de données/administratrice de base de données,2521,ESCO | ONET,3
80,web developer,développeur web/développeuse web,2513,ESCO | ONET,3
93,software developer,développeur de logiciels/développeuse de logiciels,2512,ESCO | ONET,3


In [4]:
# De-duplication: canonical skills that merged >1 source entity (e.g. Python, SQL)
cskl['n_members'] = cskl.member_entity_ids.str.split(' | ').map(len)
ms = cskl[cskl.n_members > 1].sort_values('n_members', ascending=False)
print(f'{len(ms)} multi-source canonical skills of {len(cskl)} total')
ms[['primary_label_en','primary_label_fr','hard_soft','it_subtype','sources','n_members']].head(25)

63 multi-source canonical skills of 4818 total


,primary_label_en,primary_label_fr,hard_soft,it_subtype,sources,n_members
56,Ruby (computer programming),Ruby (programmation informatique),hard,langage_ou_techno_nommee,ESCO | ONET | ROME,5
772,database management systems,systèmes de gestion de base de données,hard,donnees_bdd,ESCO | ONET | ROME,5
0,Haskell,Haskell,hard,developpement_conception,ESCO | ONET,3
78,Informatica PowerCenter,Informatica PowerCenter,hard,donnees_bdd,ESCO | ONET,3
80,Apache Tomcat,Apache Tomcat,hard,developpement_conception,ESCO | ONET,3
98,Visual Basic,Visual Basic,hard,developpement_conception,ESCO | ROME,3
58,Apache Maven,Apache Maven,hard,developpement_conception,ESCO | ONET,3
104,IBM InfoSphere DataStage,IBM InfoSphere DataStage,hard,donnees_bdd,ESCO | ONET,3
122,Java (computer programming),Java (programmation informatique),hard,langage_ou_techno_nommee,ESCO | ROME,3
140,Drupal,Drupal,hard,developpement_conception,ESCO | ONET,3


In [5]:
# Sample exactMatch alignments across sources
ex = align[align.relation=='skos:exactMatch']
print('relations:'); print(align.relation.value_counts())
ex[['source_a','source_b','confidence','method','notes']].sample(min(20, len(ex)), random_state=0)

relations:
relation
skos:relatedMatch    24466
skos:closeMatch       2103
skos:exactMatch         74
Name: count, dtype: int64


,source_a,source_b,confidence,method,notes
4059,ESCO,ONET,0.95,pref_match+embed:0.78,Pascal (computer programming) <> Pascal
5954,ESCO,ROME,0.95,pref_match+embed:0.67,Visual Basic <> Visual Basic
2469,ESCO,ONET,0.95,pref_match+embed:0.68,XQuery <> xQuery
3526,ESCO,ONET,0.95,pref_match+embed:0.73,database management systems <> Database management systems
2625,ESCO,ONET,0.95,pref_match+embed:0.68,C# <> C#
2519,ESCO,ONET,0.95,pref_match+embed:0.62,PHP <> PHP
8779,ESCO,ROME,0.95,pref_match+embed:0.58,real-time computing <> Système temps réel
4296,ESCO,ONET,0.95,pref_match+embed:0.74,Scala <> Scala
2918,ESCO,ONET,0.95,pref_match+embed:0.65,WordPress <> WordPress
1280,NOC,ONET,0.95,pref_match+embed:0.65,Data scientists <> Data Scientists


In [6]:
# Hard/soft balance and IT subtype distribution
print('hard/soft:'); print(skl.hard_soft_provisional.value_counts())
print('\nIT subtype:'); print(skl.it_subtype.value_counts())

hard/soft:
hard_soft_provisional
hard     4751
group     209
soft      132
Name: count, dtype: int64

IT subtype:
it_subtype
autre_hard                  2774
langage_ou_techno_nommee    1459
developpement_conception     248
groupe                       209
donnees_bdd                  170
soft_transversale            132
programmation                 59
securite                      32
reseau                         9
Name: count, dtype: int64
